# TODO 8 — Analyse des paramètres de la Descente de Gradient

Ce notebook applique `GradientDescent` à deux fonctions continues 2D :
- **f₁** : `x² + y²`  sur x,y ∈ [-2,+2] — paysage convexe, minimum global évident
- **f₂** : `sin(0.83x) - 1.53·cos(-1.4y) + sin(0.4·xy)` sur x,y ∈ [0,10] — paysage non-convexe

On étudie l'impact de deux paramètres :
- **α** (`learning_rate`) : taille du pas de gradient
- **δ** (`epsilon`) : pas des différences finies pour le gradient numérique

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from problem.continuous import ContinuousFunction
from solver.classical_solver.gradient_descent import GradientDescent
from utils.display_methods import plot_parameter_sweep

np.random.seed(0)
print('Imports OK')

## 1. Définition des fonctions

In [ ]:
# Fonction convexe simple
def f1(v):
    return v[0]**2 + v[1]**2

# Fonction non-convexe (multiple minima locaux)
def f2(v):
    x, y = v[0], v[1]
    return np.sin(0.83 * x) - 1.53 * np.cos(-1.4 * y) + np.sin(0.4 * x * y)

prob1 = ContinuousFunction(f1, bounds=(-2, 2),  n=2)
prob2 = ContinuousFunction(f2, bounds=(0, 10),  n=2)

print(prob1)
print(prob2)

## 2. Visualisation des paysages 2D

On affiche la heatmap de chaque fonction pour comprendre la structure du problème
avant d'appliquer la descente de gradient.

In [ ]:
def plot_landscape(fn, bounds, title, ax, n_points=200):
    lo, hi = bounds
    xs = np.linspace(lo, hi, n_points)
    ys = np.linspace(lo, hi, n_points)
    X, Y = np.meshgrid(xs, ys)
    Z = np.vectorize(lambda x, y: fn([x, y]))(X, Y)
    im = ax.contourf(X, Y, Z, levels=40, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    return X, Y, Z

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_landscape(f1, (-2, 2),  'f₁(x,y) = x²+y²  [convexe]',  axes[0])
plot_landscape(f2, (0, 10),  'f₂(x,y) = sin(0.83x)−1.53·cos(−1.4y)+sin(0.4xy)  [non-convexe]', axes[1])
plt.tight_layout()
plt.show()

## 3. Exécution sur f₁ (convexe) avec trajectoire

In [ ]:
gd1 = GradientDescent(prob1, maximize=False,
                      learning_rate=0.2, n_iterations=50,
                      domain=(-2, 2), round_solution=False)
val1, sol1, traj1 = gd1.solve(initial_solution=[1.8, -1.5], return_trajectory=True)

print(f'Solution trouvée : x={sol1[0]:.4f}, y={sol1[1]:.4f}')
print(f'Valeur           : {val1:.6f}  (optimal attendu = 0.0)')

# Affichage heatmap + trajectoire
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

X, Y, Z = plot_landscape(f1, (-2, 2), 'f₁ — trajectoire de descente', axes[0])
traj_arr = np.array(traj1)
axes[0].plot(traj_arr[:, 0], traj_arr[:, 1], 'w.-', markersize=4, linewidth=1.5, label='Trajectoire')
axes[0].plot(traj_arr[0, 0], traj_arr[0, 1], 'ro', markersize=8, label='Départ')
axes[0].plot(sol1[0], sol1[1], 'r*', markersize=12, label='Arrivée')
axes[0].legend()

energies1 = [prob1.eval(p) for p in traj1]
axes[1].plot(energies1, color='steelblue')
axes[1].set_xlabel('Itération')
axes[1].set_ylabel('f₁(x,y)')
axes[1].set_title('Convergence de f₁')

plt.tight_layout()
plt.show()

## 4. Exécution sur f₂ (non-convexe) — sensibilité au point de départ

Pour une fonction non-convexe, le résultat dépend fortement du point de départ.
On lance GD depuis plusieurs points initiaux et on observe les minima locaux atteints.

In [ ]:
N_STARTS = 12
rng = np.random.RandomState(7)
starts = rng.uniform(0, 10, size=(N_STARTS, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
X, Y, Z = plot_landscape(f2, (0, 10), 'f₂ — multiples trajectoires (non-convexe)', axes[0])

final_vals = []
colors = plt.cm.plasma(np.linspace(0.1, 0.9, N_STARTS))

for i, x0 in enumerate(starts):
    gd = GradientDescent(prob2, maximize=False,
                         learning_rate=0.05, n_iterations=300,
                         domain=(0, 10), round_solution=False)
    val, sol, traj = gd.solve(initial_solution=list(x0), return_trajectory=True)
    final_vals.append(val)
    traj_arr = np.array(traj)
    axes[0].plot(traj_arr[:, 0], traj_arr[:, 1], '-', color=colors[i], alpha=0.7, linewidth=1)
    axes[0].plot(x0[0], x0[1], 'o', color=colors[i], markersize=5)
    axes[0].plot(sol[0], sol[1], '*', color=colors[i], markersize=10)

axes[1].bar(range(N_STARTS), final_vals, color=colors)
axes[1].axhline(min(final_vals), color='red', linestyle='--', label=f'Meilleur = {min(final_vals):.3f}')
axes[1].set_xlabel('Run (point de départ différent)')
axes[1].set_ylabel('Valeur finale f₂')
axes[1].set_title('Minima locaux atteints selon le point de départ')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Valeurs finales : min={min(final_vals):.3f}, max={max(final_vals):.3f}, std={np.std(final_vals):.3f}')

## 5. Impact du learning rate α

α contrôle la taille du pas à chaque itération : x ← x − α·∇f(x).
- α trop petit : convergence très lente
- α trop grand : oscillations ou divergence (pas au-dessus du minimum)

In [ ]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 0.9]
x0_fixed = [1.8, -1.5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(alphas)))

final_vals_alpha = []
for alpha, color in zip(alphas, colors):
    gd = GradientDescent(prob1, maximize=False,
                         learning_rate=alpha, n_iterations=100,
                         domain=(-2, 2), round_solution=False)
    val, sol, traj = gd.solve(initial_solution=x0_fixed[:], return_trajectory=True)
    final_vals_alpha.append(val)
    energies = [prob1.eval(p) for p in traj]
    axes[0].plot(energies, color=color, label=f'α={alpha}')

axes[0].set_xlabel('Itération')
axes[0].set_ylabel('f₁(x,y)')
axes[0].set_title('Convergence selon α (f₁, 100 itérations)')
axes[0].set_yscale('log')
axes[0].legend(fontsize=8)

axes[1].bar([str(a) for a in alphas], final_vals_alpha, color=colors)
axes[1].set_xlabel('Learning rate α')
axes[1].set_ylabel('Valeur finale f₁')
axes[1].set_title('Valeur finale après 100 itérations selon α')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 6. Impact du pas δ (epsilon des différences finies)

Le gradient est calculé numériquement par différences finies centrées :
∂f/∂xᵢ ≈ [f(x+δeᵢ) − f(x−δeᵢ)] / (2δ)

- δ trop grand : approximation grossière du gradient (erreur de troncature)
- δ trop petit : erreurs numériques de virgule flottante (erreur d'arrondi)
- Zone optimale : δ ≈ √(ε_machine) ≈ 10⁻⁸ pour float64

In [ ]:
epsilons = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8, 1e-10]
x0_fixed = [1.5, -1.0]

final_vals_eps = []
for eps in epsilons:
    gd = GradientDescent(prob1, maximize=False,
                         learning_rate=0.1, n_iterations=200,
                         domain=(-2, 2), epsilon=eps, round_solution=False)
    val, _ = gd.solve(initial_solution=x0_fixed[:])
    final_vals_eps.append(val)
    print(f'δ={eps:.0e}  →  f₁ finale = {val:.2e}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(epsilons, final_vals_eps, 'o-', color='steelblue')
ax.set_xlabel('δ (epsilon)')
ax.set_ylabel('Valeur finale f₁')
ax.set_title('Impact de δ sur la précision du gradient (f₁, α=0.1, 200 iters)')
ax.axvline(1e-5, color='green', linestyle='--', label='δ par défaut (1e-5)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Analyse

### Cas convexe (f₁)
Sur f₁ = x²+y², GD converge vers le minimum global (0,0) quelle que soit l'initialisation,
tant que α est dans la plage valide. La convergence est linéaire (droite en échelle log).
Au-dessus d'un seuil critique (α ≈ 1/L où L est la constante de Lipschitz du gradient),
l'algorithme diverge ou oscille.

### Cas non-convexe (f₂)
Sur f₂, GD converge vers un minimum **local** qui dépend du point de départ.
La variabilité des résultats selon l'initialisation montre la limite fondamentale
de la descente de gradient pour les fonctions non-convexes.
Stratégie recommandée : **multi-start GD** (plusieurs points de départ aléatoires,
on retient le meilleur).

### Impact de α (learning rate)
- α trop petit (≤ 0.01) : convergence lente, n'atteint pas l'optimum en 100 itérations
- α optimal (≈ 0.1–0.2 pour f₁) : convergence rapide et précise
- α trop grand (≥ 0.5) : oscillations ou divergence
- Règle : α < 2/L où L = max eigenvalue du Hessien (pour f₁ : L=2, donc α < 1)

### Impact de δ (epsilon des différences finies)
Il existe une zone optimale autour de δ ∈ [10⁻⁶, 10⁻⁴] qui minimise l'erreur totale
(compromis entre erreur de troncature en O(δ²) et erreur d'arrondi en O(ε_machine/δ)).
La valeur par défaut δ=10⁻⁵ est un bon choix général pour float64.